# 第 36 课：流式音频前端总管线——顺序、状态与时间轴

把前五课组合起来，并为每个模块明确状态、延迟和失败模式。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 音频信号前端 |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 35 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | 前端顺序、模块状态、时间戳映射 |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：前端顺序、模块状态、时间戳映射。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import librosa

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"pyproject.toml").exists():return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")
ROOT=find_root();plt.rcParams["figure.figsize"]=(11,4)
print("项目根目录:",ROOT)

y,sr=sf.read(ROOT/"data"/"spoken_digits_parts"/"4_jackson_0.wav");y=y.astype(np.float32)

项目根目录: <REPO_ROOT>


## 1. 推荐概念顺序

```text
decode PCM → channel/AEC/beamforming → DC/AGC/NS
→ streaming framing → VAD/endpoint → Log-Mel/CMVN → encoder
```

实际顺序会因设备结构而变，但 AEC 必须尽早拿到同步参考，重采样和通道对齐也不能随意放置。

In [2]:
class FrontendState:
    def __init__(self,frame=200,hop=80):self.frame=frame;self.hop=hop;self.buffer=np.empty(0,np.float32);self.samples_seen=0
    def accept(self,chunk):
        chunk=np.asarray(chunk,np.float32);chunk=chunk-np.mean(chunk) # 教学版逐块去 DC
        self.buffer=np.concatenate([self.buffer,chunk]);out=[];starts=[]
        while len(self.buffer)>=self.frame:
            frame=self.buffer[:self.frame].copy();out.append(frame);starts.append(self.samples_seen)
            self.buffer=self.buffer[self.hop:];self.samples_seen+=self.hop
        return np.asarray(out),np.asarray(starts)
f=FrontendState();all_frames=[];all_starts=[]
for s in range(0,len(y),137):
    frames,starts=f.accept(y[s:s+137]);all_frames.extend(frames);all_starts.extend(starts)
print("frames",len(all_frames),"leftover samples",len(f.buffer),"last timestamp s",all_starts[-1]/sr)

frames 44 leftover samples 188 last timestamp s 0.43


## 2. 每个模块都要登记

- 输入/输出采样率与 shape；
- 每连接状态；
- 算法 lookahead；
- 时间戳如何映射；
- flush/reset 行为；
- 可观测指标；
- 旁路策略。

否则线上只看到“ASR 变差”，无法判断是 PCM、AEC、VAD、特征还是模型。

## 3. 前端质量指标

除了听感和 SNR，还应记录 clipping ratio、DC、RMS/dBFS、VAD 占比、endpoint 时长、丢包、重采样漂移，以及最终 CER/WER。

## 本课测试

1. 为什么时间戳必须从输入采样计数推导？
2. 模块 reset 遗漏会造成什么？
3. 前端降噪是否应该只用 SNR 验收？
4. VAD 应在 AEC 前还是后？
5. 旁路开关有什么价值？

<details><summary>展开参考答案</summary>

1. chunk 到达时间不等于音频内容时间。2. 上一会话状态污染下一会话。3. 不，应同时看 ASR 与分桶指标。4. 通常 AEC/增强后更可靠，但架构需结合参考同步设计。5. 快速定位模块影响并在故障时降级。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 36 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `前端顺序`、`模块状态`、`时间戳映射`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**会话 reset 遗漏导致状态串话**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**为每个模块列出 cache/flush/旁路测试**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**把麦克风前端接到第 15 课流式特征**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：前端顺序、模块状态、时间戳映射。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 前端顺序、模块状态、时间戳映射。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
